# Шарафетдинов Ринат Домашнее задание 3

In [1]:
# Импорт библиотек
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.preprocessing import LabelEncoder
# from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb

In [6]:
# Фиксация зерна для экспериментов
seed = 42

## Preprocessing after EDA

In [8]:
df_hh = pd.read_csv("/home/rinat/repos/itmo_mlsd_2024/notebooks/parsed_2024_12_28.csv", index_col=0)

In [9]:
df_hh.head()

,name,description,employer_name,salary,schedule,key_skills,experience,employment,region_name
0,PHP-разработчик (Middle),Дром даёт возможность вместе с сильной командо...,Дром,180000.0,fullDay,"{PHP,ООП,Веб-программирование,MySQL}",between3And6,full,Томская область
1,Руководитель отдела продаж (Дистрибьюция),Компания &quot;АКВАСТОК&quot; - отечественный ...,АКВАСТОК,240000.0,fullDay,"{Активные продажи,Поиск и привлечение клиентов...",between3And6,full,Москва
2,Ведущий инженер-программист,Приглашаем на работу ВЕДУЩЕГО ИНЖЕНЕРА-ПРОГРАМ...,"Энергокабель, Завод",146500.0,fullDay,"{Python,MS SQL,SQL,MS Visual Studio,C/C++}",between1And3,full,Московская область
3,Инженер-программист,ПК «Аквариус» – ведущий российский разработчик...,"Аквариус, Группа компаний",75000.0,fullDay,NaN,noExperience,full,Москва
4,Программист С#,COMITAS — первый системный интегратор междунар...,КОМИТАС,155000.0,fullDay,"{АСУ ТП,C#,C++,Автоматизация,HMI,PostgreSQL}",between1And3,full,Москва


In [10]:
df_hh.dtypes

name              object
description       object
employer_name     object
salary           float64
schedule          object
key_skills        object
experience        object
employment        object
region_name       object
dtype: object

In [11]:
df_hh.isna().sum()

name                0
description         0
employer_name       0
salary              0
schedule            0
key_skills       9605
experience          0
employment          0
region_name         0
dtype: int64

In [12]:
df_hh = df_hh.fillna("нет навыков")

In [ ]:
# Предобработка данных
encoder = LabelEncoder()
encoded_columns = ['schedule', 'experience', 'employment', 'region_name']

for column in encoded_columns:
    df_hh[column] = encoder.fit_transform(df_hh[column])
    mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))
    print(f"Mapping for {column}: {mapping}")

In [17]:
# Обработка текстовых колонок с помощью TF-IDF
text_columns = ['name', 'description', 'employer_name', 'key_skills']

def preprocess_text_columns(df, text_columns):
    for column in text_columns:
        df[column] = df[column].fillna('').astype(str).str.lower()
    return df

# Преобразование текстовых колонок в строковый формат
df_hh = preprocess_text_columns(df_hh, text_columns)

In [ ]:
skills_list = df_hh['key_skills'].tolist()

# Получаем уникальные навыки из списка
unique_skills = sorted(set(','.join(skills_list).split(',')))

# Создаем пустой датафрейм с колонками, соответствующими навыкам
skills_df = pd.DataFrame(columns=unique_skills)

# Заполняем датафрейм значениями 1 или 0, в зависимости от наличия навыка в каждом элементе списка
for skill in tqdm(unique_skills, desc="Processing skills"):
    row = [1 if skill in skills.split(',') else 0 for skills in skills_list]
    skills_df[skill] = row

In [11]:
df_hh_result = df_hh.merge(skills_df, left_index=True, right_index=True).drop(['name', 'description', 'employer_name', 'key_skills', ''], axis=1)

In [13]:
# df_hh_result.to_csv('df_hh_res.csv', index=False)

# Train Baselines

In [2]:
df_hh_result = pd.read_csv("df_hh_res.csv")

In [3]:
# Удаляем выбросы по зарплате
df_hh_result = df_hh_result.loc[(df_hh_result['salary'] > 30000) & (df_hh_result['salary'] < 1000000)].reset_index(drop=True)

In [4]:
X = df_hh_result.drop(['salary'], axis=1)
y = df_hh_result['salary']

In [ ]:
# Настройка кросс-валидации
kf = KFold(n_splits=3, shuffle=True, random_state=seed)

In [5]:
# Модель 1: Линейная регрессия
linear_model = LinearRegression()
linear_pred = cross_val_predict(linear_model, X, y, cv=kf)
linear_mape = mean_absolute_percentage_error(y, linear_pred)
print(f"Linear Regression MAPE: {linear_mape:.4f}")

# Модель 2: Random Forest
rf_model = RandomForestRegressor(random_state=seed, n_jobs=-1)
rf_pred = cross_val_predict(rf_model, X, y, cv=kf)
rf_mape = mean_absolute_percentage_error(y, rf_pred)
print(f"Random Forest MAPE: {rf_mape:.4f}")

# Модель 3: LightGBM
lgb_model = lgb.LGBMRegressor(random_state=seed, n_jobs=-1)
lgb_pred = cross_val_predict(lgb_model, X, y, cv=kf)
lgb_mape = mean_absolute_percentage_error(y, lgb_pred)
print(f"LightGBM MAPE: {lgb_mape:.4f}")

Linear Regression MAPE: 0.4231
Random Forest MAPE: 0.3255
LightGBM MAPE: 0.1783


# Grid for Best Model

In [7]:
# Грид-поиск для LightGBM
param_grid = {
    'n_estimators': [100, 200, 300, 500, 700, 1000],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1, 0.15],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=lgb.LGBMRegressor(random_state=seed, n_jobs=-1),
    param_grid=param_grid,
    cv=kf,
    scoring='neg_mean_absolute_percentage_error',
    verbose=1
)

grid_search.fit(X, y)

# Лучшие параметры
best_params = grid_search.best_params_

# Модель с лучшими параметрами
best_lgb_model = lgb.LGBMRegressor(**best_params, random_state=seed, n_jobs=-1)

# Кросс-валидация для лучшей модели
best_lgb_pred = cross_val_predict(best_lgb_model, X, y, cv=kf)
best_lgb_mape = mean_absolute_percentage_error(y, best_lgb_pred)
print(f"Best LightGBM MAPE: {best_lgb_mape:.4f}")

Best LightGBM MAPE: 0.1562


### Наилучшей моделью получился LGBMRegressor с подобранными гиперпараметрами, что позволило попасть в нужные значения метрики, которые были указаны как требования к модели (точность от 10% до 20% MAPE), скор модели равен 0.1562% MAPE. Следовательно, можно переходить к домашней работе под номером 4 и делать веб-сервис для модели